# NGSolve BEM with HTool

HTool compresses the Galerkin matrix of an NGSolve boundary integral operator without assembling the complete dense matrix. This example connects the HTool interface to the BEM operator introduced above.

In [ ]:
import subprocess
import sys

try:
    import google.colab
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    subprocess.check_call([
        "apt-get", "update", "-qq"
    ])
    subprocess.check_call([
        "apt-get", "install", "-y", "-qq",
        "cmake", "g++", "git", "libopenmpi-dev",
        "openmpi-bin", "libblas-dev", "liblapack-dev",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "ngsolve", "anywidget",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/htool-ddm/htool_python.git@beedf2f6aef52681033d19f14176f7433c5d93e8",
    ])

In [ ]:
import Htool
from matplotlib import pyplot as plt
from netgen.occ import *
from ngsolve import *
from ngsolve.bem import *
from ngsolve.webgui import Draw

## Laplace Single Layer Problem

On the unit sphere, we solve

$$
Vj=u_0|_\Gamma,
\qquad
u_0(x)=\frac{1}{|x-(1,1,1)|}.
$$

In [ ]:
sphere = Sphere((0, 0, 0), 1)
mesh = sphere.GenerateMesh(maxh=0.1).Curve(3)

fes = SurfaceL2(mesh, order=3)
u, v = fes.TnT()

u0 = 1 / sqrt((x - 1) ** 2 + (y - 1) ** 2 + (z - 1) ** 2)
rhs = LinearForm(u0 * v * ds(bonus_intorder=3)).Assemble()
V = LaplaceSL(u * ds) * v * ds

## Geometric Cluster Tree

HTool clusters degrees of freedom by position and uses their support radii to account for element size.

In [ ]:
coordinates, radii = GetDofClusteringData(fes)

cluster_tree_builder = Htool.ClusterTreeBuilder()
cluster_tree_builder.set_maximal_leaf_size(70)
cluster_tree_builder.set_partitioning_strategy(Htool.PCARegular())
cluster = cluster_tree_builder.create_cluster_tree(
    coordinates.T,
    number_of_children=2,
    radii=radii,
)

## Matrix Blocks on Demand

During compression, HTool requests submatrices

$$
A_{I,J}=(A_{ij})_{i\in I,\,j\in J}
$$

for selected row and column index sets $I$ and $J$. `CalcSubMatrixCapsule()` returns a `PyCapsule`, a Python wrapper around C++ pointers. Here it stores the callback and a reference to the NGSolve operator $V$. Python passes this capsule from NGSolve to HTool.

`GeneratorFromCapsule` opens it once. Whenever HTool needs a block,

1. HTool supplies $I$, $J$, and memory for the result,
2. the C++ callback asks NGSolve to compute exactly $A_{I,J}$,
3. NGSolve writes the values directly into HTool's memory.

NGSolve uses its usual BEM integration, including the special singular integration for touching elements. The full dense matrix is never assembled. A Python callback would require every block request to enter Python and acquire the Global Interpreter Lock (GIL). The `PyCapsule` keeps the callback in C++, allowing HTool to construct different blocks concurrently.

In [ ]:
generator = Htool.GeneratorFromCapsule(V.CalcSubMatrixCapsule())

hmatrix_builder = Htool.HMatrixTreeBuilder(
    epsilon=1e-6, eta=10, symmetry="N", UPLO="N"
)
hmatrix_builder.set_block_tree_consistency(True)
hmatrix = hmatrix_builder.build(generator, cluster, cluster)

saving = 100 * float(hmatrix.get_local_information()["Space_saving"])
print(f"H-matrix space saving: {saving:.1f}%")

## Cluster and Matrix Structure

The sphere plots show the recursive geometric partition: points with the same color belong to the same cluster at the selected depth. In the matrix plot, each rectangle represents the interaction between one target cluster and one source cluster.

- red blocks are stored densely and mainly describe nearby interactions,
- green blocks are low rank approximations of distant interactions,
- numbers inside large green blocks are their numerical ranks.

The dense blocks concentrate near the diagonal, while large parts away from it are compressed.

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax1 = fig.add_subplot(2, 2, 1, projection="3d")
ax2 = fig.add_subplot(2, 2, 2, projection="3d")
ax3 = fig.add_subplot(2, 2, 3, projection="3d")
ax4 = fig.add_subplot(2, 2, 4)

ax1.set_title("cluster tree: depth 1")
ax2.set_title("cluster tree: depth 2")
ax3.set_title("cluster tree: depth 3")
ax4.set_title("H-matrix block structure")
Htool.plot(ax1, cluster, coordinates.T, 1)
Htool.plot(ax2, cluster, coordinates.T, 2)
Htool.plot(ax3, cluster, coordinates.T, 3)
Htool.plot(ax4, hmatrix)
plt.tight_layout()
plt.show()

## Hierarchical LU Solve

The compressed matrix supports an approximate LU factorization and direct solve.

In [ ]:
hmatrix.lu_factorization()

density = GridFunction(fes)
density.vec.FV().NumPy()[:] = hmatrix.lu_solve(
    "N", rhs.vec.FV().NumPy()
)
Draw(density, mesh, order=3)